In [2]:
import pandas as pd
import glob
import os
import re
import random

# Functions

In [7]:
def process_files(files):

    dfs = []
    for file in files:
        # Leer el archivo csv
        df = pd.read_csv(file)

        #print("file: ", file)
        # 3. Extraer 'alt' y 'raan' del nombre del archivo usando expresiones regulares
        alt_match = re.search(r'alt_(\d+)', file)
        raan_match = re.search(r'raan_(\d+)', file)

        #df['alt'] = alt_match
        #df['raan'] = raan_match


        if alt_match:
            df['alt'] = int(alt_match.group(1))
        if raan_match:
            df['raan'] = int(raan_match.group(1))

        df['id_simulation'] = 'sim_alt_' +str(int(alt_match.group(1)))+ '_raan_' + str(int(raan_match.group(1)))

        ''' 
        print(df.shape)
        print('id_simulation: ', 'sim_alt_' +str(int(alt_match.group(1)))+ '_raan_' + str(int(raan_match.group(1))))
        print('0 : ', df[df['rcvOk'] == 0].shape[0], ' | ' )#,round(df[df['rcvOk'] == 0].shape[0] * 100 / df.shape[0],2))
        print('1 : ', df[df['rcvOk'] == 1].shape[0], ' | ' )#,round(df[df['rcvOk'] == 1].shape[0] * 100 / df.shape[0],2))
        '''
        dfs.append(df)

    # 4. Concatenar todos los DataFrames
    if dfs:
        final_df = pd.concat(dfs, ignore_index=True)

        # Guardar el resultado final en un nuevo archivo
        #output_filename = "Concatenated_TransmissionStatistics.csv"
        #final_df.to_csv(output_filename, index=False)

        #print(f"Se concatenaron {len(files)} archivos con éxito.")
        #print(final_df.head())
    else:
        print("No se encontraron archivos que coincidan con el patrón.")

    
    #final_df.to_csv('final_data_tranmissions.csv', index=False)
    return final_df

# Seed

In [5]:
base_path = os.path.join("results", "repetitions")

# Getting seed folders
seed_folders = sorted(glob.glob(os.path.join(base_path, "seed_*/")))
seed_folders

['results/repetitions/seed_0/',
 'results/repetitions/seed_1/',
 'results/repetitions/seed_2/',
 'results/repetitions/seed_3/',
 'results/repetitions/seed_4/',
 'results/repetitions/seed_5/',
 'results/repetitions/seed_6/',
 'results/repetitions/seed_7/',
 'results/repetitions/seed_8/',
 'results/repetitions/seed_9/']

In [ ]:
# Ruta base donde están las carpetas seed_X
base_path = os.path.join("results", "repetitions")

# Getting seed folders
seed_folders = sorted(glob.glob(os.path.join(base_path, "seed_*/")))

# Diccionarios para organizar los resultados por seed si lo necesitas
# O simplemente listas globales si quieres mezclar todas las semillas al final
all_train_pathfiles = []
all_test_pathfiles = []

# 1. Obtener todas las carpetas que coincidan con 'seed_*'
seed_folders = sorted(glob.glob(os.path.join(base_path, "seed_*/")))
print('seed_folders: ', seed_folders)
#seed_folders

for seed_folder in seed_folders:
    print(f"--- Procesando: {(seed_folder)} ---")
    
    # 2. Identificar archivos dentro de la carpeta seed actual
    file_pattern = os.path.join(seed_folder, "TransmissionStatistics_*.csv")
    files_in_seed = glob.glob(file_pattern)

    print('files_in_seed: ', len(files_in_seed))
    train_pathfiles = []
    test_pathfiles  = []

    alt_identified = []
    for file in files_in_seed:
        alt_match = re.search(r'alt_(\d+)', file)
        raan_match = re.search(r'raan_(\d+)', file)
        alt_identified.append(int(alt_match.group(1)))

    alt_identified = set(alt_identified)
    print(alt_identified)

    new_file_pattern = os.path.join(seed_folder, "TransmissionStatistics_alt*.csv")

    for alt in alt_identified:
        print(">>alt: ", alt)
        aux = "TransmissionStatistics_alt_" + str(alt) + "*.csv"
        new_file_pattern = os.path.join(seed_folder, aux)
        new_files = glob.glob(new_file_pattern)

        
        num_files_train = int(len(new_files)*0.8)
        num_files_test  = len(new_files) - num_files_train
        #print(">num_files_train: ", num_files_train)
        #print(">num_files_test: ", num_files_test)
        
        train_files_selected = random.sample(new_files, num_files_train)
        test_files_selected = [f for f in new_files if f not in train_files_selected]
        #print(">>>train_files_selected: ", train_files_selected)
        #print(">>>test_files_selected: ", test_files_selected)

        train_pathfiles.extend(train_files_selected)
        test_pathfiles.extend(test_files_selected)
        
    print(">test_files_selected: ", len(test_pathfiles), test_pathfiles)
    print(">train_pathfiles: ", len(train_pathfiles), train_pathfiles)


    train_csv = process_files(train_pathfiles)
    test_csv  = process_files(test_pathfiles)

    train_csv.to_csv(seed_folder + '_final_train_data_transmission.csv', index=False)
    #test_csv.to_csv('final_test_data_transmission.csv', index=False)


    # 1. Aseguramos el orden
    df_sorted = test_csv.sort_values(by=['id_simulation', 'time']).reset_index(drop=True)

    # 2. Calculamos el punto de corte
    split_point = len(df_sorted) // 2

    # 3. Dividimos manteniendo el orden
    test_df = df_sorted.iloc[:split_point].sort_values(by=['id_simulation', 'time']).reset_index(drop=True)
    val_df = df_sorted.iloc[split_point:].sort_values(by=['id_simulation', 'time']).reset_index(drop=True)

    test_df.to_csv(seed_folder + '_final_test_data_transmission.csv', index=False)
    val_df.to_csv(seed_folder + '_final_val_data_transmission.csv', index=False)

    print(f"Total: {len(df_sorted)} | Test: {len(test_df)} | Val: {len(val_df)}")
    

seed_folders:  ['results/repetitions/seed_0/', 'results/repetitions/seed_1/', 'results/repetitions/seed_2/', 'results/repetitions/seed_3/', 'results/repetitions/seed_4/', 'results/repetitions/seed_5/', 'results/repetitions/seed_6/', 'results/repetitions/seed_7/', 'results/repetitions/seed_8/', 'results/repetitions/seed_9/']
--- Procesando: results/repetitions/seed_0/ ---
files_in_seed:  21
{600, 850, 700, 740}
>>alt:  600
>>alt:  850
>>alt:  700
>>alt:  740
>test_files_selected:  5 ['results/repetitions/seed_0/TransmissionStatistics_alt_600_raan_170_seed_0.csv', 'results/repetitions/seed_0/TransmissionStatistics_alt_850_raan_370_seed_0.csv', 'results/repetitions/seed_0/TransmissionStatistics_alt_700_raan_170_seed_0.csv', 'results/repetitions/seed_0/TransmissionStatistics_alt_740_raan_190_seed_0.csv', 'results/repetitions/seed_0/TransmissionStatistics_alt_740_raan_140_seed_0.csv']
>train_pathfiles:  16 ['results/repetitions/seed_0/TransmissionStatistics_alt_600_raan_200_seed_0.csv', 're

# Previous

In [ ]:
file_pattern = os.path.join("results", "TransmissionStatistics_*.csv")
files = glob.glob(file_pattern)


train_pathfiles = []
test_pathfiles  = []


alt_identified = []
for file in files:
    #print(file)
    alt_match = re.search(r'alt_(\d+)', file)
    raan_match = re.search(r'raan_(\d+)', file)

    #print('alt: ', int(alt_match.group(1)))
    #print('raan: ', int(raan_match.group(1)))

    alt_identified.append(int(alt_match.group(1)))


alt_identified = set(alt_identified)
print(alt_identified)

new_file_pattern = os.path.join("results", "TransmissionStatistics_alt*.csv")

for alt in alt_identified:
    print(">>alt: ", alt)
    aux = "TransmissionStatistics_alt_" + str(alt) + "*.csv"
    new_file_pattern = os.path.join("results", aux)
    new_files = glob.glob(new_file_pattern)

    
    num_files_train = int(len(new_files)*0.8)
    num_files_test  = len(new_files) - num_files_train
    #print(">num_files_train: ", num_files_train)
    #print(">num_files_test: ", num_files_test)
    
    train_files_selected = random.sample(new_files, num_files_train)
    test_files_selected = [f for f in new_files if f not in train_files_selected]
    #print(">>>train_files_selected: ", train_files_selected)
    #print(">>>test_files_selected: ", test_files_selected)

    train_pathfiles.extend(train_files_selected)
    test_pathfiles.extend(test_files_selected)
    
print(">test_files_selected: ", len(test_pathfiles), test_pathfiles)
print(">train_pathfiles: ", len(train_pathfiles), train_pathfiles)




{600, 850, 700, 740}
>>alt:  600
>>alt:  850
>>alt:  700
>>alt:  740
>test_files_selected:  5 ['results/TransmissionStatistics_alt_600_raan_170.csv', 'results/TransmissionStatistics_alt_850_raan_370.csv', 'results/TransmissionStatistics_alt_700_raan_190.csv', 'results/TransmissionStatistics_alt_740_raan_180.csv', 'results/TransmissionStatistics_alt_740_raan_140.csv']
>train_pathfiles:  16 ['results/TransmissionStatistics_alt_600_raan_210.csv', 'results/TransmissionStatistics_alt_600_raan_200.csv', 'results/TransmissionStatistics_alt_600_raan_190.csv', 'results/TransmissionStatistics_alt_600_raan_180.csv', 'results/TransmissionStatistics_alt_850_raan_330.csv', 'results/TransmissionStatistics_alt_850_raan_350.csv', 'results/TransmissionStatistics_alt_850_raan_340.csv', 'results/TransmissionStatistics_alt_850_raan_360.csv', 'results/TransmissionStatistics_alt_700_raan_180.csv', 'results/TransmissionStatistics_alt_700_raan_200.csv', 'results/TransmissionStatistics_alt_700_raan_160.csv', 'r

{600, 850, 700, 740}
>>alt:  600
>>alt:  850
>>alt:  700
>>alt:  740
>test_files_selected:  5 ['results/TransmissionStatistics_alt_600_raan_210.csv', 'results/TransmissionStatistics_alt_850_raan_350.csv', 'results/TransmissionStatistics_alt_700_raan_160.csv', 'results/TransmissionStatistics_alt_740_raan_180.csv', 'results/TransmissionStatistics_alt_740_raan_160.csv']
>train_pathfiles:  17 ['results/TransmissionStatistics_alt_600_raan_190.csv', 'results/TransmissionStatistics_alt_600_raan_170.csv', 'results/TransmissionStatistics_alt_600_raan_200.csv', 'results/TransmissionStatistics_alt_600_raan_180.csv', 'results/TransmissionStatistics_alt_850_raan_340.csv', 'results/TransmissionStatistics_alt_850_raan_370.csv', 'results/TransmissionStatistics_alt_850_raan_330.csv', 'results/TransmissionStatistics_alt_850_raan_360.csv', 'results/TransmissionStatistics_alt_700_raan_200.csv', 'results/TransmissionStatistics_alt_700_raan_190.csv', 'results/TransmissionStatistics_alt_700_raan_170.csv', 'results/TransmissionStatistics_alt_700_raan_180.csv', 'results/TransmissionStatistics_alt_740_raan_140.csv', 'results/TransmissionStatistics_alt_740_raan_190.csv', 'results/TransmissionStatistics_alt_740_raan_210.csv', 'results/TransmissionStatistics_alt_740_raan_170.csv', 'results/TransmissionStatistics_alt_740_raan_150.csv']

In [12]:

train_csv = process_files(train_pathfiles)
test_csv  = process_files(test_pathfiles)

train_csv.to_csv('final_train_data_transmission.csv', index=False)
#test_csv.to_csv('final_test_data_transmission.csv', index=False)

(526, 23)
id_simulation:  sim_alt_600_raan_210
0 :  340  | 
1 :  186  | 
(526, 23)
id_simulation:  sim_alt_600_raan_200
0 :  337  | 
1 :  189  | 
(526, 23)
id_simulation:  sim_alt_600_raan_190
0 :  320  | 
1 :  206  | 
(526, 23)
id_simulation:  sim_alt_600_raan_180
0 :  319  | 
1 :  207  | 
(360, 23)
id_simulation:  sim_alt_850_raan_330
0 :  172  | 
1 :  188  | 
(360, 23)
id_simulation:  sim_alt_850_raan_350
0 :  161  | 
1 :  199  | 
(360, 23)
id_simulation:  sim_alt_850_raan_340
0 :  153  | 
1 :  207  | 
(360, 23)
id_simulation:  sim_alt_850_raan_360
0 :  179  | 
1 :  181  | 
(360, 23)
id_simulation:  sim_alt_700_raan_180
0 :  118  | 
1 :  242  | 
(360, 23)
id_simulation:  sim_alt_700_raan_200
0 :  154  | 
1 :  206  | 
(360, 23)
id_simulation:  sim_alt_700_raan_160
0 :  146  | 
1 :  214  | 
(360, 23)
id_simulation:  sim_alt_700_raan_170
0 :  119  | 
1 :  241  | 
(526, 23)
id_simulation:  sim_alt_740_raan_160
0 :  295  | 
1 :  231  | 
(526, 23)
id_simulation:  sim_alt_740_raan_170
0 : 

In [13]:
# 1. Aseguramos el orden
df_sorted = test_csv.sort_values(by=['id_simulation', 'time']).reset_index(drop=True)

# 2. Calculamos el punto de corte
split_point = len(df_sorted) // 2

# 3. Dividimos manteniendo el orden
test_df = df_sorted.iloc[:split_point].sort_values(by=['id_simulation', 'time']).reset_index(drop=True)
val_df = df_sorted.iloc[split_point:].sort_values(by=['id_simulation', 'time']).reset_index(drop=True)

test_df.to_csv('final_test_data_transmission.csv', index=False)
val_df.to_csv('final_val_data_transmission.csv', index=False)

print(f"Total: {len(df_sorted)} | Test: {len(test_df)} | Val: {len(val_df)}")

Total: 2298 | Test: 1149 | Val: 1149


Total: 1772 | Test: 886 | Val: 886

In [14]:
print(train_csv.shape)
print('0 : ', train_csv[train_csv['rcvOk'] == 0].shape[0], ' | ' ,round(train_csv[train_csv['rcvOk'] == 0].shape[0] * 100 / train_csv.shape[0],2))
print('1 : ', train_csv[train_csv['rcvOk'] == 1].shape[0], ' | ' ,round(train_csv[train_csv['rcvOk'] == 1].shape[0] * 100 / train_csv.shape[0],2))

(7088, 23)
0 :  3751  |  52.92
1 :  3337  |  47.08


In [7]:
test_csv['rcvOk'].value_counts()

rcvOk
0    5926
1    2770
Name: count, dtype: int64

In [15]:
print(test_csv.shape)
print('0 : ', test_csv[test_csv['rcvOk'] == 0].shape[0], ' | ' ,round(test_csv[test_csv['rcvOk'] == 0].shape[0] * 100 / test_csv.shape[0],2))
print('1 : ', test_csv[test_csv['rcvOk'] == 1].shape[0], ' | ' ,round(test_csv[test_csv['rcvOk'] == 1].shape[0] * 100 / test_csv.shape[0],2))

(2298, 23)
0 :  1359  |  59.14
1 :  939  |  40.86


# Esta es la antigua forma , hacia el random

In [27]:
# 1. Definir el patrón para encontrar los archivos
#file_pattern = "TransmissionStatistics_*.csv"
file_pattern = os.path.join("results", "TransmissionStatistics_*.csv")
files = glob.glob(file_pattern)

dfs = []

for file in files:
    # Leer el archivo csv
    df = pd.read_csv(file)

    print("file: ", file)
    # 3. Extraer 'alt' y 'raan' del nombre del archivo usando expresiones regulares
    alt_match = re.search(r'alt_(\d+)', file)
    raan_match = re.search(r'raan_(\d+)', file)

    #df['alt'] = alt_match
    #df['raan'] = raan_match


    if alt_match:
        df['alt'] = int(alt_match.group(1))
    if raan_match:
        df['raan'] = int(raan_match.group(1))

    df['id_simulation'] = 'sim_alt_' +str(int(alt_match.group(1)))+ '_raan_' + str(int(raan_match.group(1)))

    print(df.shape)
    print('0 : ', df[df['rcvOk'] == 0].shape[0], ' | ' ,round(df[df['rcvOk'] == 0].shape[0] * 100 / df.shape[0],2))
    print('1 : ', df[df['rcvOk'] == 1].shape[0], ' | ' ,round(df[df['rcvOk'] == 1].shape[0] * 100 / df.shape[0],2))

    dfs.append(df)

# 4. Concatenar todos los DataFrames
if dfs:
    final_df = pd.concat(dfs, ignore_index=True)

    # Guardar el resultado final en un nuevo archivo
    output_filename = "Concatenated_TransmissionStatistics.csv"
    final_df.to_csv(output_filename, index=False)

    print(f"Se concatenaron {len(files)} archivos con éxito.")
    print(final_df.head())
else:
    print("No se encontraron archivos que coincidan con el patrón.")

file:  results/TransmissionStatistics_alt_600_raan_190.csv
(542, 20)
0 :  330  |  60.89
1 :  212  |  39.11
file:  results/TransmissionStatistics_alt_850_raan_340.csv
(360, 20)
0 :  181  |  50.28
1 :  179  |  49.72
file:  results/TransmissionStatistics_alt_600_raan_210.csv
(542, 20)
0 :  344  |  63.47
1 :  198  |  36.53
file:  results/TransmissionStatistics_alt_850_raan_330.csv
(360, 20)
0 :  196  |  54.44
1 :  164  |  45.56
file:  results/TransmissionStatistics_alt_740_raan_180.csv
(542, 20)
0 :  344  |  63.47
1 :  198  |  36.53
file:  results/TransmissionStatistics_alt_740_raan_190.csv
(542, 20)
0 :  369  |  68.08
1 :  173  |  31.92
file:  results/TransmissionStatistics_alt_740_raan_150.csv
(542, 20)
0 :  339  |  62.55
1 :  203  |  37.45
file:  results/TransmissionStatistics_alt_600_raan_200.csv
(542, 20)
0 :  335  |  61.81
1 :  207  |  38.19
file:  results/TransmissionStatistics_alt_850_raan_360.csv
(360, 20)
0 :  199  |  55.28
1 :  161  |  44.72
file:  results/TransmissionStatistics

In [28]:
final_df.to_csv('final_data_tranmissions.csv', index=False)

In [29]:
print(final_df.shape)
final_df.head()

(7762, 20)


,pid,time,srcId,dstId,srcSat,dstSat,latDev,longDev,loraTP,loraCF,loraSF,loraBW,loraCR,satId,elevSat,doppler,rcvOk,alt,raan,id_simulation
0,31092,6.85399,42,-1,-1,-1,57.8977,24.21910,0.015849,868000000.0,12,125000,1,0,5.27902,17424.8,1,600,190,sim_alt_600_raan_190
1,31099,7.42264,47,-1,-1,-1,59.7683,8.16129,0.025119,868000000.0,11,125000,1,0,5.62503,20252.4,1,600,190,sim_alt_600_raan_190
2,31101,7.47280,7,-1,-1,-1,48.8319,14.18630,0.025119,868000000.0,12,125000,1,0,-4.56803,19677.8,0,600,190,sim_alt_600_raan_190
3,31103,7.54550,32,-1,-1,-1,50.5868,13.24900,0.025119,868000000.0,11,125000,1,0,-3.13195,19852.8,0,600,190,sim_alt_600_raan_190
4,31110,8.97020,51,-1,-1,-1,42.3684,27.90230,0.019953,868000000.0,12,125000,1,0,-8.99097,16866.5,0,600,190,sim_alt_600_raan_190


In [33]:
final_df['id_simulation'].unique()

<StringArray>
['sim_alt_600_raan_190', 'sim_alt_850_raan_340', 'sim_alt_600_raan_210',
 'sim_alt_850_raan_330', 'sim_alt_740_raan_180', 'sim_alt_740_raan_190',
 'sim_alt_740_raan_150', 'sim_alt_600_raan_200', 'sim_alt_850_raan_360',
 'sim_alt_600_raan_180', 'sim_alt_850_raan_370', 'sim_alt_740_raan_170',
 'sim_alt_850_raan_350', 'sim_alt_740_raan_160', 'sim_alt_600_raan_170',
 'sim_alt_740_raan_140']
Length: 16, dtype: str

In [30]:
print(final_df.shape)
print('0 : ', final_df[final_df['rcvOk'] == 0].shape)
print('1 : ', final_df[final_df['rcvOk'] == 1].shape)

(7762, 20)
0 :  (4809, 20)
1 :  (2953, 20)


In [31]:
print(final_df.shape)
print('0 : ', final_df[final_df['rcvOk'] == 0].shape[0], ' | ' ,round(final_df[final_df['rcvOk'] == 0].shape[0] * 100 / final_df.shape[0],2))
print('1 : ', final_df[final_df['rcvOk'] == 1].shape[0], ' | ' ,round(final_df[final_df['rcvOk'] == 1].shape[0] * 100 / final_df.shape[0],2))

(7762, 20)
0 :  4809  |  61.96
1 :  2953  |  38.04
